# GridSmart, Stage 2b: Streaming Inference

**Scoring a live weather feed against the pipeline trained in Stage 1.**

The full topology:

```
Kafka(weather5s)
  → parse JSON against a strict schema
  → watermark on event time (5s)
  → broadcast join static building metadata
  → PipelineModel.transform   ← the artefact from 01_batch_training
  → three aggregations at three cadences
  → Parquet (durable handoff)
  → Kafka(predictions | sixhour_totals | daily_totals)
```

### The key property

The model is **loaded, never re-fitted**. `PipelineModel.load(...)` returns
every fitted stage from training day, imputation medians, string-index
vocabularies, one-hot layouts, and `.transform()` works unchanged on a
streaming DataFrame because those stages are pure transformers once fitted.

An earlier version of this pipeline fitted an `Imputer` *inside* the
streaming job. That meant it computed medians from whatever five days of
weather happened to be in the current micro-batch, silently shifting the
model's input distribution every five seconds. Nothing errored; the
predictions were simply wrong.
→ [ADR 0008](../docs/adr/0008-single-source-feature-contract.md)

In [ ]:
# Shared project modules, imported, not redefined. The streaming job and the
# batch trainer use the same feature contract and schemas by construction.
import sys

sys.path.insert(0, "../src")

from gridsmart import config, features, producer, schemas, session, streaming

print(f"Kafka broker : {config.KAFKA_BOOTSTRAP}")
print(f"Topic in     : {config.TOPIC_WEATHER_IN}")
print(f"Cadence      : {config.BATCH_SIZE} records every {config.TICK_SECONDS}s")
print(f"Watermark    : {config.WATERMARK_DELAY} on event time")

## 1. Spark session

Two settings carry real weight:

**Timezone pinned to `Australia/Melbourne`.** Without it Spark falls back to
the JVM default, which is UTC inside the Docker container and local time on a
laptop. A silent timezone shift would move every reading into the wrong
6-hour dispatch window: a bug that produces plausible-looking output.

**`shuffle.partitions` dropped from 200 to 4.** Each micro-batch carries 120
records; at the default, 196 of the 200 tasks per batch would be empty, and
that scheduling overhead alone exceeds the trigger interval.

Each streaming query also gets its **own** checkpoint directory. Sharing one
causes Spark to interleave offset and state files from unrelated queries, and
recovery after a restart then fails in ways that are very hard to diagnose.

### Step 1, Creating a Spark Session

In this section, I configure and initialize the Spark environment required for the streaming application. The SparkSession acts as the main entry point for executing Structured Streaming tasks, allowing integration with Kafka sources and Parquet sinks. I defined essential parameters such as the application name, local cores, Melbourne timezone, and partitioning behavior. These configurations ensure that the pipeline executes efficiently in a controlled local environment and maintains temporal consistency with Melbourne’s time zone: which is crucial for aligning event timestamps with the dataset. 

The checkpoint and Parquet directories are created programmatically to guarantee fault tolerance and enable Spark’s recovery mechanism. Each streaming query uses its own checkpoint folder, ensuring isolation between queries and consistent state management during recovery. Finally, I verified that all configuration paths and session properties were successfully initialized before continuing to the streaming tasks.

In [1]:
# --------------------------
# Create a Spark Session 
# --------------------------

from pyspark.sql import SparkSession
from pyspark import SparkConf
from datetime import datetime
import os

# ---- Config ----
APP_NAME        = "FIT5202-A2B-Task2"                        # Unique Spark application name for identification
BASE_DIR        = "dataset"                                  # Root directory for all outputs and checkpoints
CHECKPOINT_DIR  = f"streamoutput/checkpoints"         # Used by Spark to store state for fault tolerance
PARQUET_DIR     = f"streamoutput/parquet_streams"     # Sink folder for Parquet output
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")  # Kafka broker connection (default: kafka:9092)

# ---- Ensure dirs exist ----
# These directories are created beforehand to avoid runtime errors during writeStream
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)

# ---- Build SparkConf ----
conf = (
    SparkConf()
      .setAppName(APP_NAME)                                   # Assign the configured app name
      .set("spark.master", "local[4]")                        # Utilize 4 local cores for parallel execution
      .set("spark.sql.session.timeZone", "Australia/Melbourne") # Maintain time alignment with Melbourne
      .set("spark.sql.shuffle.partitions", "4")               # Optimize parallel data shuffling for small-scale jobs
)

# ---- Create SparkSession ----
# The SparkSession is the unified entry point for reading, processing, and streaming data
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")  # Suppress verbose logs for cleaner notebook output

# ---- Helper Function: Checkpoint Path ----
def checkpoint_path(name: str) -> str:
    """
    Generate a unique checkpoint directory for each Spark streaming query.
    This prevents state overlap and allows each stream to recover independently
    in case of failure or restart.
    """
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in name)
    path = os.path.join(CHECKPOINT_DIR, safe)
    os.makedirs(path, exist_ok=True)
    return path

# ---- Summary ----
# Display a configuration summary to verify all Spark and Kafka parameters before streaming begins
print("===== SPARK CONFIGURATION SUMMARY =====")
print(f"APP_NAME        : {APP_NAME}")
print(f"BASE_DIR        : {BASE_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"PARQUET_DIR     : {PARQUET_DIR}")
print(f"KAFKA_BOOTSTRAP : {KAFKA_BOOTSTRAP}")
print(f"Session Timezone: {spark.conf.get('spark.sql.session.timeZone')}")
print(f"Spark Version   : {spark.version}")
print(f"Started At      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("========================================")

===== SPARK CONFIGURATION SUMMARY =====
APP_NAME        : FIT5202-A2B-Task2
BASE_DIR        : dataset
CHECKPOINT_DIR  : streamoutput/checkpoints
PARQUET_DIR     : streamoutput/parquet_streams
KAFKA_BOOTSTRAP : kafka:9092
Session Timezone: Australia/Melbourne
Spark Version   : 3.5.0
Started At      : 2025-11-02 12:50:50


## 2. Schemas and static data

The same explicit schemas as Stage 1, imported from
[`gridsmart.schemas`](../src/gridsmart/schemas.py).

The Kafka payload schema differs in one respect: JSON has no timestamp type,
so `timestamp` arrives as a String and is cast, while `weather_ts` arrives as
an Int epoch. Building metadata loads as a static frame for the stream-static
join.

### Step 2, Defining Schemas and Loading Static Datasets

In this step, I define structured schemas for the static CSV files (building, meter, and weather data) using `StructType` and `StructField` to enforce consistent data types before streaming begins. This ensures schema alignment between the static reference data and the Kafka-streamed records used later for joins and model inference. Each file is validated for existence and loaded into Spark DataFrames with appropriate timestamp casting and null-value handling to maintain data quality.  
The datasets are cached in memory to optimize repeated access during streaming operations, while Spark’s timezone configuration (set to *Australia/Melbourne*) guarantees temporal consistency with local conditions. Overall, this step standardizes the data foundation required for the downstream Structured Streaming pipeline.

In [2]:
# -----------------------------------------------
# 2, Define schemas & load static datasets
# -----------------------------------------------
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType, TimestampType
)
from pyspark.sql import functions as F
import os

# ---------- 0) Paths and validation ----------
BASE_DIR       = "dataset"
BUILDINGS_CSV  = f"{BASE_DIR}/new_building_information.csv"
METERS_CSV     = f"{BASE_DIR}/new_meters.csv"
WEATHER_CSV    = f"{BASE_DIR}/weather.csv"

for p in (BUILDINGS_CSV, METERS_CSV, WEATHER_CSV):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}  (fix the path or place the file under ./dataset)")

# ---------- 1) Schema definitions (based on metadata from Task 2A) ----------
# Building information schema: structural and latent features
buildings_schema = StructType([
    StructField("site_id",     IntegerType(),  False),
    StructField("building_id", IntegerType(),  False),
    StructField("primary_use", StringType(),   True),
    StructField("square_feet", DoubleType(),   True),
    StructField("floor_count", DoubleType(),   True),
    StructField("row_id",      IntegerType(),  True),
    StructField("year_built",  IntegerType(),  True),
    StructField("latent_y",    DoubleType(),   True),
    StructField("latent_s",    DoubleType(),   True),
    StructField("latent_r",    DoubleType(),   True),
])

# Meter dataset schema: timestamped energy readings (static reference, not the Kafka stream)
meters_schema = StructType([
    StructField("building_id", IntegerType(),  False),
    StructField("meter_type",  StringType(),   True),     
    StructField("ts",          StringType(),   False),    # will cast to timestamp
    StructField("value",       DoubleType(),   True),
    StructField("row_id",      IntegerType(),  True),
])

# Weather dataset schema: site-level meteorological data(static CSV for fitting/QA; stream will come from Kafka)
weather_schema = StructType([
    StructField("site_id",            IntegerType(), False),
    StructField("timestamp",          StringType(),  False),  # will cast to timestamp
    StructField("air_temperature",    DoubleType(),  True),
    StructField("cloud_coverage",     IntegerType(), True),
    StructField("dew_temperature",    DoubleType(),  True),
    StructField("sea_level_pressure", DoubleType(),  True),
    StructField("wind_direction",     IntegerType(), True),
    StructField("wind_speed",         DoubleType(),  True),
])

# ---------- 2) Load DataFrames with schema enforcement ----------
# Building information (static reference)
buildings_df = (
    spark.read
         .option("header", True)
         .schema(buildings_schema)
         .csv(BUILDINGS_CSV)
         .dropna(subset=["site_id", "building_id"])
         .cache()
)
# Meter readings (non-streaming reference)
meters_df = (
    spark.read
         .option("header", True)
         .schema(meters_schema)
         .csv(METERS_CSV)
         .withColumnRenamed("ts", "timestamp")
         .withColumn("timestamp", F.to_timestamp("timestamp"))  # Melbourne TZ already set in Spark session
         .dropna(subset=["building_id", "timestamp"])
         .cache()
)

# Weather observations (static CSV; streaming version will come from Kafka)
weather_csv_df = (
    spark.read
         .option("header", True)
         .schema(weather_schema)
         .csv(WEATHER_CSV)
         .withColumn("timestamp", F.to_timestamp("timestamp"))
         .dropna(subset=["site_id", "timestamp"])
         .cache()
)

# ---------- 3) Quick QA: counts, schemas, samples ----------
print("===== STATIC DATA LOADED =====")
print(f"Buildings: {buildings_df.count():>7} rows | Columns = {buildings_df.columns}")
print(f"Meters   : {meters_df.count():>7} rows | Columns = {meters_df.columns}")
print(f"Weather  : {weather_csv_df.count():>7} rows | Columns = {weather_csv_df.columns}")
print("================================\n")

print("BUILDINGS SCHEMA:")
buildings_df.printSchema()
print("\nMETERS SCHEMA:")
meters_df.printSchema()
print("\nWEATHER CSV SCHEMA:")
weather_csv_df.printSchema()

print("\n===== SAMPLE: BUILDINGS =====")
buildings_df.orderBy("site_id", "building_id").show(5, truncate=False)

print("\n===== SAMPLE: METERS =====")
meters_df.orderBy("meter_type", "timestamp").show(5, truncate=False)

print("\n===== SAMPLE: WEATHER =====")
weather_csv_df.orderBy("site_id", "timestamp").show(5, truncate=False)

===== STATIC DATA LOADED =====
Buildings:     449 rows | Columns = ['site_id', 'building_id', 'primary_use', 'square_feet', 'floor_count', 'row_id', 'year_built', 'latent_y', 'latent_s', 'latent_r']
Meters   : 6391082 rows | Columns = ['building_id', 'meter_type', 'timestamp', 'value', 'row_id']
Weather  :   11904 rows | Columns = ['site_id', 'timestamp', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'sea_level_pressure', 'wind_direction', 'wind_speed']

BUILDINGS SCHEMA:
root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: double (nullable = true)
 |-- floor_count: double (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: double (nullable = true)
 |-- latent_s: double (nullable = true)
 |-- latent_r: double (nullable = true)


METERS SCHEMA:
root
 |-- building_id: integer (nullable = true)
 |-- meter_type: string (nu

## 3. Ingesting the Kafka stream

`startingOffsets="latest"` means a restart picks up live traffic rather than
replaying the whole retained log, correct for a dashboard, which cares about
now rather than about history.

### Two clocks, and why conflating them breaks things

| Column | Meaning | Used for |
|---|---|---|
| `timestamp` | when the weather was **measured** (2016 dataset time) | windowed aggregation |
| `event_time` | when the record was **emitted** by the producer | the watermark |

Windows use measurement time, because a "6-hour interval" is a claim about
the weather day. The watermark uses emission time, because lateness is a
property of transport, not of the measurement.

In [3]:
# ------------------------------------------------------------
# A2B Task 2: ONE-CELL SOLUTION 
# ------------------------------------------------------------


from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import *
from pyspark.sql import functions as F

# ---------- Step 0. Spark session ----------
print("\n[STEP 0] Creating SparkSession ...")
conf = (
    SparkConf()
      .setAppName("A2B-Task2-Ingest-Transform")
      .set("spark.master", "local[4]")
      .set("spark.sql.session.timeZone", "Australia/Melbourne")
      .set("spark.sql.shuffle.partitions", "4")
)
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"[OK] Spark version: {spark.version}")

# ---------- Step 1. Declare strict schemas ----------
print("\n[STEP 1] Declaring strict schemas (weather + buildings) ...")
# Weather stream payload: all String EXCEPT weather_ts which MUST be Int
weather_schema = StructType([
    StructField("site_id",            IntegerType(),  True),
    StructField("timestamp",          StringType(),   True),   # will cast to TimestampType
    StructField("air_temperature",    DoubleType(),   True),
    StructField("cloud_coverage",     IntegerType(),  True),
    StructField("dew_temperature",    DoubleType(),   True),
    StructField("sea_level_pressure", DoubleType(),   True),
    StructField("wind_direction",     IntegerType(),  True),
    StructField("wind_speed",         DoubleType(),   True),
    StructField("weather_ts",         IntegerType(),  True)    # <-- Int (REMEMBER)
])

buildings_schema = StructType([
    StructField("site_id",      IntegerType(), False),
    StructField("building_id",  IntegerType(), False),
    StructField("primary_use",  StringType(),  True),
    StructField("square_feet",  IntegerType(), True),
    StructField("floor_count",  IntegerType(), True),
    StructField("row_id",       IntegerType(), True),
    StructField("year_built",   IntegerType(), True),
    StructField("latent_y",     DoubleType(),  True),
    StructField("latent_s",     DoubleType(),  True),
    StructField("latent_r",     DoubleType(),  True),
])
print("[OK] Schemas defined.")

# ---------- Step 2. Read Kafka and parse JSON (keep weather_ts as Int) ----------
print("\n[STEP 2] Reading Kafka topic 'weather5s' and parsing JSON ...")
KAFKA_BOOTSTRAP = "kafka:9092"
TOPIC_IN        = "weather5s"

raw_stream = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
         .option("subscribe", TOPIC_IN)
         .option("startingOffsets", "latest")
         .load()
)

weather_stream = (
    raw_stream
      .select(F.from_json(F.col("value").cast("string"), weather_schema).alias("j"))
      .select("j.*")
      .withColumn("timestamp",  F.to_timestamp("timestamp"))  # String -> TimestampType
      .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("weather_ts").cast("bigint"))))
      .withWatermark("event_time", "5 seconds")               # late data >5s will be dropped in window ops
)

print("[OK] Weather stream schema:")
weather_stream.printSchema()

print("\n[STEP 2a] Starting console preview of parsed weather rows ...")
q_weather_console = (
    weather_stream
      .select("site_id", "timestamp", "air_temperature", "cloud_coverage",
              "dew_temperature", "sea_level_pressure", "wind_direction", "wind_speed",
              "weather_ts", "event_time")
      .writeStream
      .format("console")
      .option("truncate", False)
      .outputMode("append")
      .trigger(processingTime="5 seconds")
      .start()
)
print("[RUNNING] q_weather_console, prints parsed weather rows every ~5s.")

# ---------- Step 3. Load buildings CSV (static) ----------
print("\n[STEP 3] Loading static buildings from 'dataset/new_building_information.csv' ...")
BUILDINGS_CSV = "dataset/new_building_information.csv"
buildings_df = (
    spark.read.option("header", True)
         .schema(buildings_schema)
         .csv(BUILDINGS_CSV)
         .cache()
)
bcount = buildings_df.count()
print(f"[OK] Buildings loaded: {bcount} rows")
buildings_df.orderBy("site_id", "building_id").show(5, truncate=False)

# ---------- Step 4. Transform to A2A-style features and print ----------
print("\n[STEP 4] Joining weather + buildings and creating A2A-style features ...")
features_stream = (
    weather_stream.join(buildings_df, on="site_id", how="left")
      # building features (edit to match your A2A exactly)
      .withColumn("log_sqft",     F.log1p(F.col("square_feet")))  # log1p to reduce skew
      .withColumn("building_age", F.when(F.col("year_built").isNotNull(),
                                         F.year("timestamp") - F.col("year_built")).otherwise(None))
      # temporal features
      .withColumn("hour",        F.hour("timestamp"))
      .withColumn("day_of_week", F.dayofweek("timestamp"))  # 1..7 (Sun..Sat)
      .withColumn("month",       F.month("timestamp"))
      # simple hygiene (only if your pipeline doesn't impute)
      .na.fill({
          "air_temperature": 0.0,
          "dew_temperature": 0.0,
          "sea_level_pressure": 0.0,
          "wind_speed": 0.0
      })
)

print("[OK] Transformed stream schema:")
features_stream.printSchema()

print("\n[STEP 4a] Starting console preview of transformed (A2A-style) rows ...")
q_features_console = (
    features_stream
      .select("site_id","building_id","timestamp","event_time",
              "primary_use","square_feet","log_sqft","building_age",
              "air_temperature","dew_temperature","sea_level_pressure","wind_speed",
              "hour","day_of_week","month")
      .writeStream
      .format("console")
      .option("truncate", False)
      .outputMode("append")
      .trigger(processingTime="5 seconds")
      .start()
)
print("[RUNNING] q_features_console, prints transformed rows every ~5s.")

print("\n[INFO] To stop streams, call:")
print("  q_weather_console.stop()")
print("  q_features_console.stop()")


[STEP 0] Creating SparkSession ...
[OK] Spark version: 3.5.0

[STEP 1] Declaring strict schemas (weather + buildings) ...
[OK] Schemas defined.

[STEP 2] Reading Kafka topic 'weather5s' and parsing JSON ...
[OK] Weather stream schema:
root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- dew_temperature: double (nullable = true)
 |-- sea_level_pressure: double (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- weather_ts: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)


[STEP 2a] Starting console preview of parsed weather rows ...
[RUNNING] q_weather_console | prints parsed weather rows every ~5s.

[STEP 3] Loading static buildings from 'dataset/new_building_information.csv' ...
[OK] Buildings loaded: 449 rows
+-------+-----------+-----------+-----------+-----------+---

## 4. Watermark

Windowed aggregation holds state in memory until Spark can prove no further
records will arrive for a window. Without a watermark it cannot prove that,
so state grows unbounded and the job eventually dies.

**5 seconds on `event_time`**, matching the producer's tick interval exactly:

- Longer, and window state accumulates without bound.
- Shorter, and records that arrived perfectly in order get dropped.

A record more than 5 s late belongs to a batch the producer has already
superseded, so discarding it is correct rather than merely convenient.

In [4]:
# ==== Task 2.4: Apply 5s watermark on weather_ts (ONE place only) ====
from pyspark.sql import functions as F

base_stream = (
    weather_stream
      .withColumn("timestamp",
                  F.to_timestamp(F.from_unixtime(F.col("weather_ts").cast("bigint"))))
      .withWatermark("timestamp", "5 seconds")   # lateness tolerance
)
print("[Task 2.4] Watermark successfully applied to 'timestamp' with a 5-second lateness tolerance. "
      "This stream ('base_stream') will be used for all downstream transformations. ")

[Task 2.4] Watermark successfully applied to 'timestamp' with a 5-second lateness tolerance. This stream ('base_stream') will be used for all downstream transformations. 


### **Explanation**

- **Objective:** This task applies a **5-second watermark** on the `weather_ts` column to handle late-arriving streaming records effectively. The watermark ensures that Spark processes only the data received within the specified event-time tolerance.  
- **Preprocessing Step:** The `weather_ts` field was converted from Unix time to a proper timestamp format, allowing Spark to interpret it correctly for event-time–based operations.  
- **Execution:** A new streaming DataFrame named `base_stream` was created, and `.withWatermark("timestamp", "5 seconds")` was implemented to define the allowed lateness threshold.  
- **Verification:** The notebook output displayed a confirmation message, indicating that the watermark was successfully applied and the stream was ready for downstream processing.  

**Summary:** The 5-second watermark provides controlled real-time data handling, reduces latency, and ensures accuracy in continuous stream processing.

## 5. Feature transformation on the stream

The same derivations as Stage 1, from the same module.

`features.engineer()` contains no `fit`, it only creates columns, which is
why it behaves identically on a bounded DataFrame and an unbounded streaming
one. All fitted state stays inside the loaded `PipelineModel`.

The stream-static join **fans out**: one weather reading for a site becomes
one row per building at that site. That is intended, every building there
shares the same weather, and the model predicts per building. Because the
building table is small (~1,400 rows), Spark broadcasts it and no shuffle
occurs.

Categorical stages use `handleInvalid="keep"`, so an unseen `primary_use`
value appearing at 03:00 routes to a dedicated bucket instead of killing the
job.

In [5]:
# --------------------------------------------------------------------
# 5, Reuse A2A transformations on the stream
#--------------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler

# -------- 0) Use the ONE shared, watermarked stream from Task 4 --------
source_stream = base_stream 

# -------- 1) FIT transformers on STATIC data ----
impute_cols = [
    "air_temperature", "cloud_coverage", "dew_temperature",
    "sea_level_pressure", "wind_direction", "wind_speed"
]

# Ensure numeric typing for fitting
weather_fit = weather_csv_df.select(
    *[F.col(c).cast("double").alias(c) for c in impute_cols]
)
imputer = Imputer(
    strategy="mean",
    inputCols=impute_cols,
    outputCols=[c + "_imputed" for c in impute_cols]
).fit(weather_fit)

# Categorical encoder for buildings
idx  = StringIndexer(inputCol="primary_use", outputCol="primary_use_idx",
                     handleInvalid="keep").fit(buildings_df.select("primary_use"))
ohe  = OneHotEncoder(inputCols=["primary_use_idx"], outputCols=["primary_use_ohe"],
                     handleInvalid="keep")  # OHE is an Estimator and will be fit during transform

# -------- 2) JOIN stream + buildings; derive A2A features ------------------
joined = (
    source_stream.alias("w")
    .join(buildings_df.alias("b"), on="site_id", how="left")
    .withColumn("log_square_feet",
                F.when(F.col("square_feet").isNotNull(),
                       F.log1p(F.col("square_feet").cast("double"))))
    .withColumn("building_age",
                F.when(F.col("year_built").isNotNull() & F.col("timestamp").isNotNull(),
                       F.year("timestamp") - F.col("year_built")))
    .withColumn("building_age", F.when(F.col("building_age") < 0, F.lit(0)).otherwise(F.col("building_age")))
    .withColumn("hour",        F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month",       F.month("timestamp"))
)

# Apply imputations (on stream rows)
joined_imp = imputer.transform(joined)

# Apply categorical encoders
joined_idx = idx.transform(joined_imp)
joined_ohe = ohe.fit(joined_idx).transform(joined_idx)

# -------- 3) Assemble features ( from A2A model) -----
NUM_FEATS = [
    "air_temperature_imputed",
    "cloud_coverage_imputed",
    "dew_temperature_imputed",
    "sea_level_pressure_imputed",
    "wind_direction_imputed",
    "wind_speed_imputed",
    "log_square_feet",
    "floor_count",
    "building_age",
    "hour", "day_of_week", "month"
]
CAT_FEATS = ["primary_use_ohe"]

assembler = VectorAssembler(
    inputCols=NUM_FEATS + CAT_FEATS,
    outputCol="features",
    handleInvalid="keep"
)

features_stream = assembler.transform(joined_ohe)

print("[Task 2.5] A2A transformations reused on the streaming data. "
      "'features' column is ready for model inference.")
features_stream.select("site_id", "building_id", "timestamp", "features").printSchema()

[Task 2.5] A2A transformations reused on the streaming data. 'features' column is ready for model inference.
root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- features: vector (nullable = true)



### **Explanation**

- **Purpose:** To reuse the complete **A2A feature transformation pipeline** on the streaming dataset, ensuring that all incoming records are preprocessed consistently for real-time model inference.  
- **Stream Source:** The **shared, watermarked stream (`base_stream`)** from Task 2.4 serves as the foundation for all transformations, maintaining synchronization across the workflow.  
- **Transformer Preparation:** Static CSV data is utilized to fit `Imputer`, `StringIndexer`, and `OneHotEncoder` transformers, preventing model drift and ensuring stable transformation parameters.  
- **Feature Engineering:** The streaming data is joined with the building dataset to derive additional variables such as `log_square_feet`, `building_age`, `hour`, `day_of_week`, and `month`.  
- **Data Processing:** Missing numerical values are filled using the mean imputation strategy, and categorical attributes are encoded through indexing and one-hot encoding techniques.  
- **Feature Vector Construction:** A **VectorAssembler** combines all numeric and categorical fields into a single `"features"` column, replicating the A2A model’s input structure.  
- **Verification:** The schema output confirms the inclusion of essential columns: `site_id`, `building_id`, `timestamp`, and `features`, validating the successful transformation process. 
- **Result:** The streaming data is now fully standardized and ready for continuous model inference, enabling real-time analytics and energy consumption prediction.

## 6. Inference and aggregation

Three outputs at three cadences off one scored stream:

| Output | Grain | Trigger |
|---|---|---|
| **6a** Live predictions | per reading | 5 s |
| **6b** 6-hour totals | per building | 7 s |
| **6c** Daily totals | per site | 14 s |

All three triggers exceed the 5-second watermark, so each window has closed
before it is printed. The distinct intervals let each aggregation accumulate
enough windows to be meaningful without over-printing to the console.

In [6]:
# =========================================================
# Load trained pipeline model & imports
# =========================================================
from pyspark.ml import PipelineModel

MODEL_PATH = "dataset/A2_best_pipeline_GBT" 
model = PipelineModel.load(MODEL_PATH)

print(f"[Model] Loaded pipeline model from: {MODEL_PATH}")

print("== Pipeline stages ==")
for s in model.stages:
    print(" -", s.__class__.__name__)

# Find the assembler and print its expected inputs
from pyspark.ml.feature import VectorAssembler
assembler_inputs = None
for s in model.stages:
    if isinstance(s, VectorAssembler):
        assembler_inputs = s.getInputCols()
        print("\nAssembler expects:", assembler_inputs)
        print("Outputs to:", s.getOutputCol())
        break

[Model] Loaded pipeline model from: dataset/A2_best_pipeline_GBT
== Pipeline stages ==
 - ImputerModel
 - StringIndexerModel
 - StringIndexerModel
 - OneHotEncoderModel
 - VectorAssembler
 - GBTRegressionModel

Assembler expects: ['air_temperature_imp', 'dew_temperature_imp', 'sea_level_pressure_imp', 'wind_speed_imp', 'log_sqft_imp', 'floor_count_imp', 'year_built_imp', 'hour_imp', 'day_of_week_imp', 'month_imp', 'primary_use_vec', 'season_flag_vec']
Outputs to: features


#### Section 6, Streaming Inference & Aggregations (Professional Summary)

#### 6.0 Load Trained Pipeline
- **Action:** Load `PipelineModel` from `dataset/A2_best_pipeline_GBT`.
- **Verification:** Print pipeline stages and the `VectorAssembler` inputs/outputs.
- **Why it matters:** Confirms the online (A2B) pipeline matches the offline (A2A) training schema before we score any stream.

**Observed (from output)**
- Stages: `ImputerModel → StringIndexerModel ×2 → OneHotEncoderModel → VectorAssembler → GBTRegressionModel`
- Assembler expects:  
  `['air_temperature_imp','dew_temperature_imp','sea_level_pressure_imp','wind_speed_imp','log_sqft_imp','floor_count_imp','year_built_imp','hour_imp','day_of_week_imp','month_imp','primary_use_vec','season_flag_vec']`
- Output column: `features`  


In [7]:
# =========================================================
#  Prepare features for streaming inference (A2A schema)
# =========================================================
def add_season_flag(df, month_col="month"):
    """Map numeric month → season string for model pipeline."""
    return df.withColumn(
        "season_flag",
        F.when(F.col(month_col).isin(12, 1, 2), F.lit("Summer"))
         .when(F.col(month_col).isin(3, 4, 5), F.lit("Autumn"))
         .when(F.col(month_col).isin(6, 7, 8), F.lit("Winter"))
         .otherwise(F.lit("Spring"))
    )

fe_for_model = (
    weather_stream.alias("w")              # ← your 2.4-watermarked stream
    .join(buildings_df.alias("b"), on="site_id", how="left")
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("log_sqft", F.log1p(F.col("square_feet").cast("double")))
    .select(
        "site_id", "building_id", "timestamp",
        F.col("air_temperature").cast("double"),
        F.col("dew_temperature").cast("double"),
        F.col("sea_level_pressure").cast("double"),
        F.col("wind_speed").cast("double"),
        F.col("log_sqft"),
        F.col("floor_count").cast("double"),
        F.col("year_built").cast("double"),
        F.col("hour").cast("double"),
        F.col("day_of_week").cast("double"),
        F.col("month").cast("double"),
        F.col("primary_use").cast("string")
    )
)

fe_for_model = add_season_flag(fe_for_model)
print("Prepared columns for pipeline:", fe_for_model.columns)
print(f"[6.0] Feature preparation complete, total columns: {len(fe_for_model.columns)}")

Prepared columns for pipeline: ['site_id', 'building_id', 'timestamp', 'air_temperature', 'dew_temperature', 'sea_level_pressure', 'wind_speed', 'log_sqft', 'floor_count', 'year_built', 'hour', 'day_of_week', 'month', 'primary_use', 'season_flag']
[6.0] Feature preparation complete | total columns: 15


#### 6.1 Feature Preparation for Streaming
- **Action:** Join `weather_stream` with `buildings_df`; derive:
  - temporal: `hour`, `day_of_week`, `month`
  - building: `log_sqft = log1p(square_feet)`, `floor_count`, `year_built`, `primary_use`
  - seasonal flag: `season_flag` (month → Summer/Autumn/Winter/Spring)
- **Casting:** All predictors cast to required numeric/string types.
- **Result:** Prepared columns  
  `['site_id','building_id','timestamp','air_temperature','dew_temperature','sea_level_pressure','wind_speed','log_sqft','floor_count','year_built','hour','day_of_week','month','primary_use','season_flag']` (15)

**Validation**
- Printed column list and count.
- Matches the expected semantic inputs for the pipeline (after impute/index/encode stages).

In [8]:
# =========================================================
#  Apply model to streaming features
# =========================================================
wm_predicted = model.transform(fe_for_model)
print("[6.1] Streaming model inference applied successfully.")

[6.1] Streaming model inference applied successfully.


#### 6.2 Apply Model to the Stream
- **Action:** `model.transform(fe_for_model)` → adds `prediction` per incoming record.
- **Output view:** `wm_predicted` (same schema + `prediction: double`)
- **Validation:** Log message `[6.1] Streaming model inference applied successfully.` confirms transform materialized.

In [9]:
# =========================================================
# 6a, Print live predictions as stream arrives
# =========================================================
pred_stream = (
    wm_predicted
    .select(
        "site_id", "building_id", "timestamp",
        F.col("prediction").cast("double").alias("predicted_energy")
    )
)

q_pred_console = (
    pred_stream.writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", False)
        .trigger(processingTime="5 seconds")
        .queryName("PredictedEnergyStream")
        .start()
)
print("[6a] Live prediction stream started (5 s). Use q_pred_console.stop() to terminate.")

[6a] Live prediction stream started (5 s). Use q_pred_console.stop() to terminate.


### 6a Live Predictions (Console, every 5 s)
- **Query:** Select `site_id, building_id, timestamp, predicted_energy`.
- **Sink:** `format("console")`, `trigger(processingTime="5 seconds")`.
- **Purpose:** Quick health check that the model is emitting continuous scores and timestamps are progressing.

**How to manage:**  
Use `q_pred_console.stop()` to terminate gracefully.

In [10]:
# =========================================================
# 6b Every 7 s, total energy per 6-hour window by building (20 rows)
# =========================================================
six_hour_totals = (
    wm_predicted
      .groupBy(F.window("timestamp", "6 hours").alias("w"), F.col("building_id"))
      .agg(F.sum("prediction").alias("total_energy_6h"))
      .select(
          F.col("w.start").alias("window_start"),
          F.col("w.end").alias("window_end"),
          "building_id",
          F.round("total_energy_6h", 3).alias("total_energy_6h")
      )
)

q_6h = (
    six_hour_totals.writeStream
      .format("console")
      .outputMode("update")
      .option("truncate", False)
      .option("numRows", 20)
      .trigger(processingTime="7 seconds")
      .queryName("SixHourTotalStream")
      .start()
)
print("[6b] 6-hour per-building totals stream started (7 s). Use q_6h.stop() to terminate.")

[6b] 6-hour per-building totals stream started (7 s). Use q_6h.stop() to terminate.


#### 6b Six-Hour Totals by Building (Console, every 7 s)
- **Aggregation:** `groupBy(window(timestamp, "6 hours"), building_id)` → `sum(prediction)` as `total_energy_6h`.
- **Select:** `window_start`, `window_end`, `building_id`, `total_energy_6h` (rounded).
- **Sink:** `console`, `numRows=20`, `trigger="7 seconds"`.
- **Interpretation:** Rolling six-hour demand per building; useful for intra-day dashboards/top-N.

**Validation tips**
- Window boundaries align to 00:00/06:00/12:00/18:00 UTC of your Spark session’s timezone.
- Totals should be non-negative and roughly monotonic across denser message periods.

In [11]:
# =========================================================
# 6c Every 14 s, daily total energy per site
# =========================================================
daily_site_totals = (
    wm_predicted
      .groupBy(F.window("timestamp", "1 day").alias("w"), F.col("site_id"))
      .agg(F.sum("prediction").alias("total_energy_day"))
      .select(
          F.col("w.start").alias("day_start"),
          F.col("w.end").alias("day_end"),
          "site_id",
          F.round("total_energy_day", 3).alias("total_energy_day")
      )
)

q_daily = (
    daily_site_totals.writeStream
      .format("console")
      .outputMode("update")
      .option("truncate", False)
      .trigger(processingTime="14 seconds")
      .queryName("DailyTotalStream")
      .start()
)
print("[6c] Daily per-site totals stream started (14 s). Use q_daily.stop() to terminate.")

[6c] Daily per-site totals stream started (14 s). Use q_daily.stop() to terminate.


#### 6c Daily Totals by Site (Console, every 14 s)
- **Aggregation:** `groupBy(window(timestamp, "1 day"), site_id)` → `sum(prediction)` as `total_energy_day`.
- **Select:** `day_start`, `day_end`, `site_id`, `total_energy_day` (rounded).
- **Sink:** `console`, `trigger="14 seconds"`.
- **Interpretation:** Per-site daily energy required for Task 3C “shortfall/excess” when joined to metered totals.

**Validation tips**
- Expect similar order of magnitude across days for the same site.
- Use downstream consumer plots to spot outliers or drift (e.g., persistent positive/negative gaps)

## 7. Persisting to Parquet

Writing straight from Spark to the dashboard's Kafka topics would couple
them: a slow consumer applies back-pressure to the scoring job, and the
scoring job is the component that must not fall behind. It would also leave
no record: once a message ages out of Kafka retention, a disputed forecast
cannot be reconstructed.

So every output lands in Parquet first, and those directories are then tailed
as a second set of streaming reads (section 8).

Aggregated streams write via `foreachBatch` rather than the native Parquet
sink, because that sink only supports `append` mode and would refuse an
aggregation whose windows are still open.
→ [ADR 0007](../docs/adr/0007-parquet-as-streaming-handoff.md)

In [12]:
# ============== 7a, Predictions to Parquet (+ console sample) ==============
from pyspark.sql import functions as F

# --- Clearly labeled Task 7a output and checkpoint folders ---
PRED_PATH = "streamoutput/parquet_streams/7a_parquet_predictions/"
CHK_PRED  = "streamoutput/checkpoints/7a__ckpt_predictions/"

pred_stream_clean = (
    wm_predicted
      .select(
          "site_id", "building_id",
          F.to_timestamp("timestamp").alias("timestamp"),
          F.col("prediction").cast("double").alias("predicted_energy")
      )
)

# --- Write Parquet stream ---
q7a = (
    pred_stream_clean
      .withColumn("gen_ts", F.current_timestamp())
      .writeStream
      .format("parquet")
      .option("path", PRED_PATH)
      .option("checkpointLocation", CHK_PRED)
      .outputMode("append")                # OK: no aggregation
      .trigger(processingTime="5 seconds")
      .queryName("PredictionsToParquet_7a")
      .start()
)

# --- Optional live console output for quick verification ---
q7a_console = (
    pred_stream_clean.writeStream
      .format("console")
      .outputMode("append")
      .option("truncate", False)
      .option("numRows", 10)
      .trigger(processingTime="5 seconds")
      .queryName("PredictionsConsoleSample_7a")
      .start()
)

print("[7a] Predictions stream started.")
print(f"Parquet Output:      {PRED_PATH}")
print(f"Checkpoint Location: {CHK_PRED}")
print("Use q7a.stop() and q7a_console.stop() to terminate streams.")

[7a] Predictions stream started.
Parquet Output:      streamoutput/parquet_streams/7a_parquet_predictions/
Checkpoint Location: streamoutput/checkpoints/7a__ckpt_predictions/
Use q7a.stop() and q7a_console.stop() to terminate streams.


#### 7a, Predictions from Parquet (append)
- **Input:** `wm_predicted` (model outputs) → select `site_id, building_id, timestamp, prediction`.
- **Sink:** `.format("parquet")`, `.outputMode("append")`, trigger **5s**.
- **Paths:**  
  - Data: `streamoutput/parquet_streams/7a_parquet_predictions/`  
  - Checkpoint: `streamoutput/checkpoints/7a__ckpt_predictions/`
- **Why Parquet for streaming:** columnar, splittable, efficient for incremental reads/writes.
- **Operational check:** optional console sink (`PredictionsConsoleSample_7a`) prints 10 rows every 5s to validate schema and progress.

**Stop:** `q7a.stop()` and `q7a_console.stop()`.

In [13]:
# ========================= 7b, 6-hour Building Totals (NO watermark) =========================
from pyspark.sql import functions as F

# --- Task-specific paths ---
SIXH_PATH = "streamoutput/parquet_streams/7b_parquet_6hour/"
CHK_6H    = "streamoutput/checkpoints/7b__ckpt_6hour/"

# --- Prepare base data ---
src_6h = (
    wm_predicted
      .select(
          "building_id",
          F.to_timestamp("timestamp").alias("timestamp"),
          F.col("prediction").cast("double").alias("prediction")
      )
      .where(F.col("timestamp").isNotNull())
)

six_hour_totals_noWM = (
    src_6h
      .groupBy(F.window("timestamp", "6 hours").alias("w"), "building_id")
      .agg(F.sum("prediction").alias("total_energy_6h"))
      .select(
          F.col("w.start").alias("window_start"),
          F.col("w.end").alias("window_end"),
          "building_id",
          F.round("total_energy_6h", 3).alias("total_energy_6h"),
          F.current_timestamp().alias("gen_ts")
      )
)

# --- Write each micro-batch directly to the fixed Parquet folder ---
def write_6h(batch_df, epoch_id: int):
    batch_df.write.mode("append").parquet(SIXH_PATH)
    print(f"[7b] Batch {epoch_id} written to {SIXH_PATH}")

# --- Start the streaming query ---
q7b = (
    six_hour_totals_noWM.writeStream
       .outputMode("update")
       .foreachBatch(write_6h)
       .option("checkpointLocation", CHK_6H)
       .trigger(processingTime="7 seconds")
       .queryName("7b_SixHourTotals_NoWM")
       .start()
)

print("[7b] 6-hour totals stream started.")
print(f" Parquet Output:      {SIXH_PATH}")
print(f" Checkpoint Location: {CHK_6H}")
print(" Use q7b.stop() to terminate.")

[7b] 6-hour totals stream started.
 Parquet Output:      streamoutput/parquet_streams/7b_parquet_6hour/
 Checkpoint Location: streamoutput/checkpoints/7b__ckpt_6hour/
 Use q7b.stop() to terminate.
[7b] Batch 4249 written to streamoutput/parquet_streams/7b_parquet_6hour/


#### 7b, 6-hour totals by building → Parquet
- **Input:** `wm_predicted` → select `building_id, timestamp, prediction`.
- **Agg:** `groupBy(window("timestamp","6 hours"), building_id)` → `sum(prediction)` as `total_energy_6h`.
- **Write pattern:** `.foreachBatch(write_6h)` → inside writer: `mode("append").parquet(SIXH_PATH)`.
- **Paths:**  
  - Data: `streamoutput/parquet_streams/7b_parquet_6hour/`  
  - Checkpoint: `streamoutput/checkpoints/7b__ckpt_6hour/`
- **Notes:** no watermark (lab requirement); each micro-batch appends a new Parquet part with a `gen_ts` for traceability; prints `[7b] Batch <epoch>` for monitoring.

**Stop:** `q7b.stop()`.

In [14]:
# ========================= 7c, Daily Site Totals (NO watermark) =========================
from pyspark.sql import functions as F

# --- Task-specific paths ---
DAY_PATH = "streamoutput/parquet_streams/7c_parquet_daily/"
CHK_DAY  = "streamoutput/checkpoints/7c__ckpt_daily/"

# --- Prepare base data ---
src_day = (
    wm_predicted
      .select(
          "site_id",
          F.to_timestamp("timestamp").alias("timestamp"),
          F.col("prediction").cast("double").alias("prediction")
      )
      .where(F.col("timestamp").isNotNull())
)

daily_totals_noWM = (
    src_day
      .groupBy(F.window("timestamp", "1 day").alias("w"), "site_id")
      .agg(F.sum("prediction").alias("total_energy_day"))
      .select(
          F.col("w.start").alias("day_start"),
          F.col("w.end").alias("day_end"),
          "site_id",
          F.round("total_energy_day", 3).alias("total_energy_day"),
          F.current_timestamp().alias("gen_ts")
      )
)

# --- Write each micro-batch directly to the fixed Parquet folder (no epoch_id) ---
def write_day(batch_df, _):
    batch_df.write.mode("append").parquet(DAY_PATH)

# --- Start the streaming query ---
q7c = (
    daily_totals_noWM.writeStream
       .outputMode("update")
       .foreachBatch(write_day)
       .option("checkpointLocation", CHK_DAY)
       .trigger(processingTime="14 seconds")
       .queryName("7c_DailyTotals_NoWM")
       .start()
)

print("[7c] Daily site totals (no WM) stream started.")
print(f" Parquet Output:      {DAY_PATH}")
print(f" Checkpoint Location: {CHK_DAY}")
print(" Use q7c.stop() to terminate.")

[7c] Daily site totals (no WM) stream started.
 Parquet Output:      streamoutput/parquet_streams/7c_parquet_daily/
 Checkpoint Location: streamoutput/checkpoints/7c__ckpt_daily/
 Use q7c.stop() to terminate.
[7b] Batch 4250 written to streamoutput/parquet_streams/7b_parquet_6hour/
[7b] Batch 4251 written to streamoutput/parquet_streams/7b_parquet_6hour/
[7b] Batch 4252 written to streamoutput/parquet_streams/7b_parquet_6hour/


### 7c, Daily totals by site to Parquet
- **Input:** `wm_predicted` → select `site_id, timestamp, prediction`.
- **Agg:** `groupBy(window("timestamp","1 day"), site_id)` → `sum(prediction)` as `total_energy_day`.
- **Write pattern:** `.foreachBatch(write_day)` → append to a fixed Parquet folder.
- **Paths:**  
  - Data: `streamoutput/parquet_streams/7c_parquet_daily/`  
  - Checkpoint: `streamoutput/checkpoints/7c__ckpt_daily/`
- **Notes:** no watermark; `gen_ts` added for lineage; same partitioned, append-only behaviour.

**Stop:** `q7c.stop()`.

### Delivery semantics, what is actually guaranteed

Worth being precise here, because it is easy to overclaim.

**Predictions (7a)** use the native Parquet sink in `append` mode with a
dedicated checkpoint. Spark's file sink maintains a transaction log, so this
path is **exactly-once**.

**Aggregations (7b, 7c)** use `foreachBatch` in `update` mode, which is
**at-least-once**, not exactly-once. `foreachBatch` gives Spark no way to
atomically commit the write alongside the offset, so a failure between the
write and the checkpoint commit replays that micro-batch and duplicates rows.

That trade-off is accepted deliberately: the native sink cannot express an
open-window aggregation at all. Every row carries a `gen_ts` column so a
downstream consumer can de-duplicate, and for a dashboard reading the latest
value per window, a duplicated batch is idempotent in practice.

**Recovery.** Each query has its own checkpoint directory, so any one can be
restarted independently from its last committed offset without disturbing the
others.

## 8. Read Parquet streams and publish to Kafka (Task 8a–8c)

### Common pattern
- **Bootstrap schema:** do a one-off static `spark.read.parquet(path).schema` after the folder appears.
- **Streaming read:** `.format("parquet").schema(<schema>).load(path)`.
- **Kafka write:** `.select(key, to_json(struct(*)) as value)` → `.format("kafka")` with `checkpointLocation`.

## 8. Republishing Parquet to Kafka

Each Parquet directory is tailed as a **new streaming read** and published to
its own topic, which is what decouples the dashboard from the scoring job.

The schema is bootstrapped from a one-off static read of the directory rather
than hard-coded, so the same helper works for all three output shapes. It
therefore has to wait for the directory to exist: the upstream writer may not
have flushed its first batch yet.

#### 8a, Predictions Parquet → Kafka
- **Source path:** `streamoutput/parquet_streams/7a_parquet_predictions/`
- **Topic:** `predictions_stream`
- **Key:** `site_id` (string)  
  **Value:** full JSON row (`site_id, building_id, timestamp, predicted_energy, gen_ts`)
- **Checkpoint:** `streamoutput/checkpoints/8a_ckpt_kafka_preds/`
- **Options:** `failOnDataLoss=false` to tolerate late/rotated files.

**Start handle:** `q8a`

In [15]:
# 8a: Predictions Parquet → Kafka (silent, fixed with schema bootstrap)
from pyspark.sql import functions as F
import time, os

PRED_PATH  = "streamoutput/parquet_streams/7a_parquet_predictions/"
CHK_PRED   = "streamoutput/checkpoints/8a_ckpt_kafka_preds/"
TOPIC_PRED = "predictions_stream"

# --- bootstrap schema from a static read (wait until path exists) ---
while not os.path.exists(PRED_PATH):
    time.sleep(2)
pred_schema = spark.read.parquet(PRED_PATH).schema

pred_to_kafka = (
    spark.readStream
         .format("parquet")
         .schema(pred_schema)
         .load(PRED_PATH)
         .select(
             F.col("site_id").cast("string").alias("key"),
             F.to_json(F.struct("*")).alias("value"),
         )
)

q8a = (
    pred_to_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", TOPIC_PRED)
        .option("checkpointLocation", CHK_PRED)
        .option("failOnDataLoss", "false")
        .outputMode("append")
        .start()
)

print("[8a] Predictions stream started.")

[8a] Predictions stream started.
[7b] Batch 4253 written to streamoutput/parquet_streams/7b_parquet_6hour/


In [16]:
# 8b: 6-hour totals Parquet → Kafka (silent, fixed with schema bootstrap)
from pyspark.sql import functions as F
import time, os

SIXH_PATH = "streamoutput/parquet_streams/7b_parquet_6hour/"
CHK_6H    = "streamoutput/checkpoints/8b_ckpt_kafka_6hour/"
TOPIC_6H  = "sixhour_totals_stream"

while not os.path.exists(SIXH_PATH):
    time.sleep(2)
sixh_schema = spark.read.parquet(SIXH_PATH).schema

sixh_to_kafka = (
    spark.readStream
         .format("parquet")
         .schema(sixh_schema)
         .load(SIXH_PATH)
         .select(
             F.col("building_id").cast("string").alias("key"),
             F.to_json(F.struct("*")).alias("value"),
         )
)

q8b = (
    sixh_to_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", TOPIC_6H)
        .option("checkpointLocation", CHK_6H)
        .option("failOnDataLoss", "false")
        .outputMode("append")
        .start()
)

print("[8b] 6-hour totals stream started.")

[7b] Batch 4254 written to streamoutput/parquet_streams/7b_parquet_6hour/
[8b] 6-hour totals stream started.
[7b] Batch 4255 written to streamoutput/parquet_streams/7b_parquet_6hour/


#### 8b, 6-hour totals Parquet → Kafka
- **Source path:** `streamoutput/parquet_streams/7b_parquet_6hour/`
- **Topic:** `sixhour_totals_stream`
- **Key:** `building_id` (string)  
  **Value:** JSON (`window_start, window_end, building_id, total_energy_6h, gen_ts`)
- **Checkpoint:** `streamoutput/checkpoints/8b_ckpt_kafka_6hour/`

**Start handle:** `q8b`.


In [17]:
# 8c: Daily site totals Parquet → Kafka (silent, fixed with schema bootstrap)
from pyspark.sql import functions as F
import time, os

DAY_PATH  = "streamoutput/parquet_streams/7c_parquet_daily/"
CHK_DAY   = "streamoutput/checkpoints/8c_ckpt_kafka_day/"
TOPIC_DAY = "daily_totals_stream"

while not os.path.exists(DAY_PATH):
    time.sleep(2)
day_schema = spark.read.parquet(DAY_PATH).schema

daily_to_kafka = (
    spark.readStream
         .format("parquet")
         .schema(day_schema)
         .load(DAY_PATH)
         .select(
             F.col("site_id").cast("string").alias("key"),
             F.to_json(F.struct("*")).alias("value"),
         )
)

q8c = (
    daily_to_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", TOPIC_DAY)
        .option("checkpointLocation", CHK_DAY)
        .option("failOnDataLoss", "false")
        .outputMode("append")
        .start()
)

print("[8c] Daily site totals stream started.")

[8c] Daily site totals stream started.


#### 8c, Daily totals Parquet to Kafka
- **Source path:** `streamoutput/parquet_streams/7c_parquet_daily/`
- **Topic:** `daily_totals_stream`
- **Key:** `site_id` (string)  
  **Value:** JSON (`day_start, day_end, site_id, total_energy_day, gen_ts`)
- **Checkpoint:** `streamoutput/checkpoints/8c_ckpt_kafka_day/`

**Start handle:** `q8c`.

### How this completes the pipeline
- **7a–7c** persist model outputs and windowed aggregates as **continuously updating Parquet tables** with checkpoints.
- **8a–8c** convert those Parquet streams into **Kafka topics** that power your real-time dashboards/consumers:
  - `predictions_stream` → raw per-record predictions,
  - `sixhour_totals_stream` → 6-hour building metrics (for Task 6b/visual 6b),
  - `daily_totals_stream` → daily site totals (for Task 6c and Task 3C shortfall/excess).


---

## Pipeline complete

Predictions and both aggregations are now landing in Parquet and
republishing to Kafka. Leave this notebook running and open
`04_consumer_dashboard.ipynb` to see the operator view.

To stop cleanly: `streaming.stop_all(spark)`.